# 01 · Fase 1 (equipe): grade de configurações léxicas

Pré-requisito: notebook 00 homologado e OpenSearch 3.9.0 rodando.
Este notebook não altera o `01_fase1.ipynb` original. Ele:
1. roda várias configurações BM25 (cada uma com experimento, índice e pasta novos);
2. grava um `comparison.csv` limpo (uma linha por configuração);
3. compara consulta a consulta;
4. monta a pasta de entrega (com correção do `opensearch_version`), verifica e compacta.

**Altere `TEAM` antes de executar.** Para repetir tudo do zero, mude `TAG` (índices/pastas existentes nunca são apagados).

In [1]:
from pathlib import Path
import sys, json, shutil, csv, subprocess, zipfile
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "ri_core.py").is_file()), None)
assert ROOT is not None, "Abra o notebook a partir da pasta do pacote."
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from ri_core import load_data, load_qrels, load_run, evaluate, write_json, write_csv
from ri_workflow import fresh
from os_retrieval import run_experiment, OpenSearchHTTP, validate_config
print("Raiz do pacote:", ROOT)

Raiz do pacote: /home/vtexlab/Downloads/lab_ri_opensearch_Fase_01/lab_ri_opensearch


## Parâmetros

Grade: E1 isola o analyzer `standard`; E2 é o mesmo BM25 padrão (k1=1.2, b=0.75) com `portuguese`
(stopwords + stemmer); E3–E6 variam k1 (saturação de frequência) e b (normalização por tamanho) sobre `portuguese`.
Os textos têm mediana de 38 palavras, mas cauda até ~1.000 — por isso vale testar b menor e maior.

In [2]:
TEAM = "equipe_16"          # <-- ALTERE para o ID atribuído (ex.: equipe_07)
TAG = "v1"                   # mude para v2, v3... se quiser refazer tudo com nomes novos

EXPERIMENTS = [
    # nome (só a-z 0-9 _ -),  analyzer,     k1,  b
    ("std_k12_b075",          "standard",   1.2, 0.75),
    ("pt_k12_b075",           "portuguese", 1.2, 0.75),
    ("pt_k09_b040",           "portuguese", 0.9, 0.40),
    ("pt_k15_b075",           "portuguese", 1.5, 0.75),
    ("pt_k12_b050",           "portuguese", 1.2, 0.50),
    ("pt_k12_b090",           "portuguese", 1.2, 0.90),
]
BASELINE = "pt_k12_b075"     # referência para a comparação consulta a consulta

import re
assert re.fullmatch(r"equipe_[0-9]{2}", TEAM) and TEAM != "equipe_XX", "Defina TEAM com o ID da equipe"
DATA = ROOT / "data/dev"
RES = ROOT / "resultados"
BASE_CONFIG = json.loads((ROOT / "starter/config.json").read_text(encoding="utf-8"))

def make_config(analyzer, k1, b):
    c = dict(BASE_CONFIG, team_id=TEAM, mode="lexical", analyzer=analyzer, bm25_k1=k1, bm25_b=b)
    return validate_config(c)

print(OpenSearchHTTP().check()["version"]["number"])

3.9.0


## Execução da grade (reexecutar é seguro: experimentos já concluídos são reaproveitados)

In [3]:
docs, queries = load_data(DATA)
qrels = load_qrels(DATA / "qrels.csv", docs, queries)
results = {}
for name, analyzer, k1, b in EXPERIMENTS:
    exp = f"{name}_{TAG}"
    out = RES / f"{TEAM}_{exp}"
    index = f"ri-{TEAM}-{exp}"
    config = make_config(analyzer, k1, b)
    if not (out / "metrics.json").is_file():
        print("Executando", exp, "->", index)
        run_experiment(DATA, out, config, index)
        _, ranking = load_run(out / "run.csv", docs, queries)
        per_query, summary = evaluate(ranking, queries, qrels)
        write_json(out / "metrics.json", summary)
        write_csv(out / "per_query.csv", list(per_query[0]), per_query)
    else:
        print("Reaproveitando", exp)
    summary = json.loads((out / "metrics.json").read_text(encoding="utf-8"))
    per_query = {r["query_id"]: r for r in csv.DictReader((out / "per_query.csv").open(encoding="utf-8"))}
    results[name] = dict(exp=exp, out=out, config=config, summary=summary, per_query=per_query)
print("Concluído:", len(results), "configurações")

Executando std_k12_b075_v1 -> ri-equipe_16-std_k12_b075_v1
Executando pt_k12_b075_v1 -> ri-equipe_16-pt_k12_b075_v1
Executando pt_k09_b040_v1 -> ri-equipe_16-pt_k09_b040_v1
Executando pt_k15_b075_v1 -> ri-equipe_16-pt_k15_b075_v1
Executando pt_k12_b050_v1 -> ri-equipe_16-pt_k12_b050_v1
Executando pt_k12_b090_v1 -> ri-equipe_16-pt_k12_b090_v1
Concluído: 6 configurações


## comparison.csv e comparação consulta a consulta

In [4]:
FIELDS = ["experiment", "analyzer", "bm25_k1", "bm25_b", "ndcg@10", "recall@100", "mrr@10"]
COMPARISON = RES / f"{TEAM}_comparison_{TAG}.csv"
rows = []
for name, r in results.items():
    s, c = r["summary"], r["config"]
    rows.append({"experiment": r["exp"], "analyzer": c["analyzer"], "bm25_k1": c["bm25_k1"], "bm25_b": c["bm25_b"],
                 **{k: round(s[k], 4) for k in ["ndcg@10", "recall@100", "mrr@10"]}})
write_csv(COMPARISON, FIELDS, rows)   # sobrescreve: sempre uma linha por configuração

def paired(a, b, metric="ndcg@10", eps=1e-9):
    pa, pb = results[a]["per_query"], results[b]["per_query"]
    d = [float(pa[q][metric]) - float(pb[q][metric]) for q in pa]
    return sum(x > eps for x in d), sum(x < -eps for x in d)

print(f"{'experimento':<22}{'ndcg@10':>9}{'rec@100':>9}{'mrr@10':>8}   vs {BASELINE} (ndcg: melhora/piora)")
for row, name in zip(rows, results):
    w, l = paired(name, BASELINE)
    print(f"{name:<22}{row['ndcg@10']:>9.4f}{row['recall@100']:>9.4f}{row['mrr@10']:>8.4f}   +{w} / -{l}")
print("\nArquivo:", COMPARISON)

experimento             ndcg@10  rec@100  mrr@10   vs pt_k12_b075 (ndcg: melhora/piora)
std_k12_b075             0.7745   0.9667  0.7575   +1 / -5
pt_k12_b075              0.8513   0.9667  0.8256   +0 / -0
pt_k09_b040              0.8579   0.9667  0.8348   +2 / -2
pt_k15_b075              0.8640   0.9667  0.8426   +2 / -0
pt_k12_b050              0.8471   0.9667  0.8198   +1 / -2
pt_k12_b090              0.8539   0.9667  0.8389   +1 / -1

Arquivo: /home/vtexlab/Downloads/lab_ri_opensearch_Fase_01/lab_ri_opensearch/resultados/equipe_16_comparison_v1.csv


## Consultas difíceis (rank do relevante no baseline vs. melhor configuração)
Útil para a seção de dificuldades do README.

In [5]:
best = max(results, key=lambda n: (results[n]["summary"]["ndcg@10"], results[n]["summary"]["recall@100"]))
def rank_of_relevant(name, qid):
    rel = {d for d, v in qrels[qid].items() if v > 0}
    with (results[name]["out"] / "run.csv").open(encoding="utf-8") as f:
        for r in csv.DictReader(f):
            if r["query_id"] == qid and r["doc_id"] in rel:
                return int(r["rank"])
    return None   # fora do top-100
print(f"{'query_id':<9}{BASELINE:>14}{best:>14}  consulta")
for qid in sorted(queries, key=lambda q: rank_of_relevant(BASELINE, q) or 999):
    a, b = rank_of_relevant(BASELINE, qid), rank_of_relevant(best, qid)
    print(f"{qid:<9}{str(a):>14}{str(b):>14}  {queries[qid]['text'][:70]}")

query_id    pt_k12_b075   pt_k15_b075  consulta
33053                 1             1  O Neon54 tem uma aplicação para telemóvel?
123422                1             1  A Avenida Suplementos é realmente confiável?
198973                1             1  Quanto tempo faz One Wolf Os códigos promocionais duram?
74810                 1             1  Consegue integrar e-commerce no meu site de impressão?
203068                1             1  As piscinas Intex são duráveis?
166209                1             1  O The Beaufort disponibiliza Wi-Fi grátis?
118542                1             1  Redimin é seguro para todos os tipos de pessoas?
94587                 1             1  Posso assistir a The Office online gratuitamente?
152357                1             1  Quais taxas são cobradas por meio do cartão de crédito?
73063                 1             1  Se eu não sou suíço , posso utilizar esses sites?
84752                 1             1  Qual melhor lugar para comprar roupa em São

## Escolha final
Critério: NDCG@10 (desempate Recall@100). Com 30 consultas e um único relevante por consulta,
uma consulta move a média em ~0,03; só abandonamos o padrão `pt_k12_b075` se o ganho for
claro **e** distribuído (mais consultas melhorando que piorando). Deixe `FINAL = None` para usar a sugestão.

In [6]:
def suggestion():
    base = results[BASELINE]["summary"]
    cands = []
    for n, r in results.items():
        gain = r["summary"]["ndcg@10"] - base["ndcg@10"]
        w, l = paired(n, BASELINE)
        if n == BASELINE or (gain >= 0.02 and w > l and r["summary"]["recall@100"] >= base["recall@100"] - 1e-9):
            cands.append((r["summary"]["ndcg@10"], r["summary"]["recall@100"], n))
    return max(cands)[2]

FINAL = None   # ou o nome de um experimento, ex.: "pt_k12_b075"
FINAL = FINAL or suggestion()
print("Configuração final:", FINAL, results[FINAL]["config"], results[FINAL]["summary"])

Configuração final: pt_k12_b075 {'team_id': 'equipe_16', 'analyzer': 'portuguese', 'bm25_k1': 1.2, 'bm25_b': 0.75, 'title_boost': 1.0, 'mode': 'lexical', 'candidate_k': 100, 'rrf_k': 60} {'judged@10': 0.09333333333333334, 'missing_queries': 0, 'mrr@10': 0.8255555555555555, 'ndcg@10': 0.8512687091380094, 'queries': 30, 'recall@100': 0.9666666666666667}


## Montar a entrega
Correção aplicada: o `os_retrieval.py` copiado para a entrega lê o `config.json` **da entrega** como `DEFAULTS`
e exige `opensearch_version` em `check()`. Sem essa chave, `run.py` falha com `KeyError`.

In [7]:
DELIVERY = ROOT / "entregas" / f"{TEAM}_phase1_solution"
fresh(DELIVERY)
shutil.copy2(ROOT / "starter/run.py", DELIVERY / "run.py")
for module in ["os_retrieval.py", "ri_core.py", "ri_workflow.py"]:
    shutil.copy2(ROOT / module, DELIVERY / module)
final_config = dict(results[FINAL]["config"], opensearch_version="3.9.0")
write_json(DELIVERY / "config.json", final_config)
(DELIVERY / "requirements-extra.txt").write_text("", encoding="utf-8")
(DELIVERY / "resultados_dev").mkdir()
shutil.copy2(COMPARISON, DELIVERY / "resultados_dev/comparison.csv")

table = "\n".join(f"| {r['experiment']} | {r['analyzer']} | {r['bm25_k1']} | {r['bm25_b']} | {r['ndcg@10']:.4f} | {r['recall@100']:.4f} | {r['mrr@10']:.4f} |" for r in rows)
cmd = f"python entregas/{TEAM}_phase1_solution/run.py --data data/dev --out resultados/check_fase1 --index ri-{TEAM}-check-fase1"
readme = f'''# {TEAM.replace("_", " ").title()} — Fase 1 (busca léxica BM25)

**ID:** {TEAM}

**Integrantes:** [preencher]

## Comando de execução (a partir da raiz do pacote, OpenSearch 3.9.0 em 127.0.0.1:9200)

```
{cmd}
```

Gera `run.csv` e `run.meta.json` na pasta de `--out` (que deve ser nova/vazia). O índice de `--index` deve começar com `ri-` e não existir. Dependências: apenas `requirements.txt` do pacote (`requirements-extra.txt` vazio).

## Configurações comparadas (dev: 10 mil documentos, 30 consultas, 1 relevante por consulta)

| experimento | analyzer | k1 | b | NDCG@10 | Recall@100 | MRR@10 |
|---|---|---|---|---|---|---|
{table}

[preencher: justificativa de cada variação]

## Decisão final

`{FINAL}` — analyzer `{final_config["analyzer"]}`, k1={final_config["bm25_k1"]}, b={final_config["bm25_b"]}, consulta `multi_match` em `title`/`text`, top-100.

[preencher: justificativa]

## Dificuldades conhecidas

- O `os_retrieval.py` copiado para a pasta da entrega lê o `config.json` da equipe como configuração padrão e exige `opensearch_version`; incluímos `"opensearch_version": "3.9.0"` no `config.json` para o `run.py` funcionar.
- [preencher]
'''
(DELIVERY / "README_EQUIPE.md").write_text(readme, encoding="utf-8")
print("Entrega montada em", DELIVERY)
print(sorted(p.relative_to(DELIVERY).as_posix() for p in DELIVERY.rglob("*") if p.is_file()))

Entrega montada em /home/vtexlab/Downloads/lab_ri_opensearch_Fase_01/lab_ri_opensearch/entregas/equipe_16_phase1_solution
['README_EQUIPE.md', 'config.json', 'os_retrieval.py', 'requirements-extra.txt', 'resultados_dev/comparison.csv', 'ri_core.py', 'ri_workflow.py', 'run.py']


## Verificar com o script oficial (cria índice `ri-verify-...` e pasta temporária)

In [8]:
r = subprocess.run([sys.executable, str(ROOT / "verify_submission.py"), "--solution", str(DELIVERY)],
                   cwd=ROOT, capture_output=True, text=True)
print(r.stdout[-3000:], r.stderr[-3000:])
assert r.returncode == 0, "Verificação falhou"


OK: solução reproduzida; 30 consultas; índice ri-verify-8d2463d2385e
 


## Compactar (rode só depois de preencher integrantes e justificativas no README)

In [9]:
readme_text = (DELIVERY / "README_EQUIPE.md").read_text(encoding="utf-8")
assert "[preencher" not in readme_text, "Complete o README_EQUIPE.md antes de compactar"
ZIP = ROOT / "entregas" / f"{TEAM}_fase1.zip"
assert not ZIP.exists(), f"{ZIP} já existe; apague ou renomeie conscientemente"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(DELIVERY.rglob("*")):
        if p.is_file() and "__pycache__" not in p.parts:
            z.write(p, p.relative_to(DELIVERY).as_posix())
with zipfile.ZipFile(ZIP) as z:
    names = z.namelist()
required = {"run.py", "config.json", "README_EQUIPE.md", "requirements-extra.txt", "resultados_dev/comparison.csv",
            "os_retrieval.py", "ri_core.py", "ri_workflow.py"}
assert required <= set(names), required - set(names)
assert not any(n.endswith((".jsonl", "qrels.csv")) for n in names)
print(ZIP, names)

AssertionError: Complete o README_EQUIPE.md antes de compactar